# Concept Embedding as a Prompt Token

Steerling decomposes hidden states into **concept embeddings** (4096-d vectors, same
space as token embeddings). Can we inject one directly into the prompt and steer
what the model talks about?

**Idea:** Build a normal chat prompt `"Tell me about [CONCEPT_EMB]"`, where
`[CONCEPT_EMB]` is a concept embedding swapped into the token embedding table.
The model never saw concept embeddings as input tokens during training.

**Key finding:** Concepts with high **top-token alignment** (how strongly the
embedding projects onto real token directions via lm_head) can be vocabulized in this way.

**Requirements:** GPU with >= 18 GB VRAM

## 1. Setup

In [60]:
import torch
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig, ConceptCatalog

model = AutoModel.from_pretrained("guidelabs/steerling-8b-instruct", trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("guidelabs/steerling-8b-instruct", trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")
catalog = ConceptCatalog.load()

## 2. Prompt & Helpers

We insert a **placeholder token** (`<|bos|>`) into the user message, then swap its
embedding row with a concept embedding before generation. Because `tok_emb` and
`lm_head` share weights (weight tying), we also ban the placeholder from output
to avoid artifacts.

In [62]:
USER_MSG = "Tell me about "
PLACEHOLDER_ID = tokenizer.bos_token_id
ALIGNMENT_THRESHOLD = 0.85

# Build prompt with a placeholder slot for the concept embedding
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
    {"role": "user", "content": USER_MSG},
]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
prompt_ids = tokenizer.encode(formatted, add_special_tokens=False)

eoc_id = tokenizer.endofchunk_token_id
eoc_positions = [i for i, t in enumerate(prompt_ids) if t == eoc_id]
insert_pos = eoc_positions[1]  # before user turn's <|endofchunk|>
prompt_ids_with_concept = prompt_ids[:insert_pos] + [PLACEHOLDER_ID] + prompt_ids[insert_pos:]
prompt_tensor = torch.tensor([prompt_ids_with_concept], dtype=torch.long, device=generator.device)

eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
config = GenerationConfig(
    max_new_tokens=128, steps=128, temperature=0.4, seed=42, stop_tokens=[eot_id],
)

SEP = "#" * 80
print(f"Prompt: \"{USER_MSG}[CONCEPT]\"")
print(f"Alignment threshold: {ALIGNMENT_THRESHOLD}")

Prompt: "Tell me about [CONCEPT]"
Alignment threshold: 0.85


In [63]:
def get_top_alignment(emb):
    """Return the highest LM-head alignment score for this embedding."""
    emb_unit = emb.float() / (emb.float().norm() + 1e-12)
    return (generator.model.transformer.lm_head.weight.float() @ emb_unit).max().item()


def generate_with_concept(concept_emb):
    """Swap placeholder embedding, generate, restore."""
    tok_emb = generator.model.transformer.tok_emb
    original = tok_emb.weight.data[PLACEHOLDER_ID].clone()
    tok_emb.weight.data[PLACEHOLDER_ID] = concept_emb

    # Ban placeholder from output (weight tying makes lm_head row change too)
    original_banned = generator._banned_ids.clone()
    generator._banned_ids = torch.cat([
        original_banned, torch.tensor([PLACEHOLDER_ID], dtype=torch.long),
    ])
    try:
        out = generator.generate_full(prompt_tensor, config)
    finally:
        tok_emb.weight.data[PLACEHOLDER_ID] = original
        generator._banned_ids = original_banned
    return out.text


def concept_info(emb, concept_obj=None):
    """Print embedding diagnostics."""
    emb_f = emb.float()
    emb_unit = emb_f / (emb_f.norm() + 1e-12)
    alignment = generator.model.transformer.lm_head.weight.float() @ emb_unit
    values, indices = alignment.topk(8)
    top_str = ", ".join(f"{tokenizer.decode([int(i)])}({v:.2f})" for i, v in zip(indices, values))
    info = f"  norm: {emb_f.norm().item():.2f} | top-1 alignment: {values[0].item():.2f}"
    if concept_obj is not None:
        info += f" | steerable: {concept_obj.is_steerable}"
    print(info)
    print(f"  top tokens: {top_str}")

## 3. Baseline (no concept)

Plain `"Tell me about "` with no concept embedding — the model has nothing to latch
onto and gives a generic greeting.

In [64]:
baseline = generator.generate(formatted, config)
print(SEP)
print("BASELINE (no concept embedding)")
print(SEP)
print(baseline)

################################################################################
BASELINE (no concept embedding)
################################################################################
Hello! I'm here to help you with information and assistance. I'm your friendly, knowledgeable AI assistant. You can ask me questions on a wide range of topics, from science and history to pop culture and everyday life. Whether you're looking for fun facts, advice, or just want to chat, feel free to share your interests or. Let's get started


## 4. Generate with Known Concept Embeddings

5 known concepts selected with **top-1 alignment >= 0.90** and
**norm >= 3.0**. Each concept's embedding is swapped in as the placeholder token.

In [65]:
# Selected via scan: top-1 alignment >= 0.90, norm >= 3.0, diverse topics
CONCEPT_IDS = [
    363,    # Animals and Creatures         (top1=1.68, norm=3.32)
    27140,  # Spaceflight and Space Exploration (top1=1.13, norm=4.61)
    14609,  # Sugar Production and Refining (top1=1.37, norm=3.18)
    18254,  # Dentistry and Oral Health     (top1=1.25, norm=4.15)
    33392,  # Wind and blowing motion       (top1=1.20, norm=3.78)
]

for cid in CONCEPT_IDS:
    c = catalog.get(cid)
    emb = generator.model.known_head._get_embedding(
        torch.tensor([cid], device=generator.device)
    )[0]

    print(SEP)
    print(f"{c.name}  (id={cid})")
    print(SEP)
    concept_info(emb, c)
    text = generate_with_concept(emb)
    print(f"  output: {text[:300]}")
    print()

################################################################################
Animals and Creatures  (id=363)
################################################################################
  norm: 3.32 | top-1 alignment: 1.68 | steerable: False
  top tokens:  animal(1.68),  animals(1.55),  Animal(1.50), animal(1.49), Animal(1.43), animals(1.40),  Animals(1.39), Anim(1.25)


  output: Certainly! Here’s an overview of **animal**:

### What is an Animal?

An animal is a living organism that belongs to the kingdom **Animalia**, characterized by:
- Multicellular eukaryotic cells
- Ability for locomotion (movement)
- Sensory organs (eyes, ears, nose, tongue, skin)
- Metabolism (the ab

################################################################################
Spaceflight and Space Exploration  (id=27140)
################################################################################
  norm: 4.61 | top-1 alignment: 1.13 | steerable: True
  top tokens:  astronauts(1.13),  astronaut(1.07),  spacecraft(0.99), onaut(0.97),  NASA(0.97),  Apollo(0.97),  SpaceX(0.94),  Shuttle(0.93)
  output: Certainly! Here’s an overview of **Space**:

### What is Space?
- **Space** is the region outside Earth’s atmosphere, beyond our planet and its moon.
- It includes stars, planets, moons, comets, asteroids, meteoroids, and interstellar space (the void between stars).
- Astro